# 1. Análise Exploratória e Descritiva dos Dados

## Objetivo

Este notebook realiza uma análise exploratória completa das bases de dados IACOV-BR, com foco em entender:
- A estrutura e dimensões dos dados
- A distribuição de valores faltantes
- As características demográficas dos pacientes
- As variáveis clínicas e laboratoriais
- A distribuição da variável alvo (óbito)
- Possíveis padrões e anomalias nos dados

## Por que isso é importante?

Antes de construir qualquer modelo de aprendizado de máquina, é essencial entender bem os dados. Esta análise inicial nos ajuda a:
1. **Identificar problemas**: Valores faltantes, outliers, inconsistências
2. **Tomar decisões informadas**: Sobre limpeza, transformação e seleção de variáveis
3. **Evitar erros**: Garbage in, garbage out - dados ruins produzem modelos ruins
4. **Gerar hipóteses**: Sobre relações entre variáveis que exploraremos depois

---

## Seção 1: Importações e Configurações

In [ ]:
# Importações de manipulação de dados
import pandas as pd
import numpy as np
from pathlib import Path

# Importações de visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Importações estatísticas
from scipy import stats

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configurações de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.2f}' if abs(x) < 1e6 else f'{x:.2e}')

print("✓ Todas as bibliotecas importadas com sucesso!")

## Seção 2: Carregamento de Dados

Vamos carregar ambas as bases de dados e fazer uma comparação inicial.

In [ ]:
# Definir caminhos dos dados
data_dir = Path("../data/raw")
raw_file = data_dir / "F_Tabela_Geral_Final(1).xlsx"
treated_file = data_dir / "df_iacov_treated.xlsx"

# Carregar dados
print("Carregando base de dados bruta...")
df_raw = pd.read_excel(raw_file)
print(f"✓ Base bruta carregada: {df_raw.shape[0]:,} linhas × {df_raw.shape[1]} colunas\n")

print("Carregando base de dados tratada...")
df_treated = pd.read_excel(treated_file)
print(f"✓ Base tratada carregada: {df_treated.shape[0]:,} linhas × {df_treated.shape[1]} colunas")

## Seção 3: Visão Geral da Base Bruta

### O que é a base bruta?
A base bruta contém todos os dados originais coletados do IACOV-BR, sem qualquer tratamento ou seleção de variáveis. Ela é mais completa em termos de variáveis, mas contém mais valores faltantes.

In [ ]:
print("="*80)
print("BASE DE DADOS BRUTA - INFORMAÇÕES GERAIS")
print("="*80)

print(f"\nDimensões: {df_raw.shape[0]:,} pacientes × {df_raw.shape[1]} variáveis")
print(f"\nPrimeiras 5 linhas:")
print(df_raw.head())

In [ ]:
# Informações sobre tipos de dados
print("\nTipos de dados:")
print(df_raw.dtypes.value_counts())

print("\nColunas por tipo:")
print(f"  - Numéricas: {df_raw.select_dtypes(include=[np.number]).shape[1]}")
print(f"  - Texto: {df_raw.select_dtypes(include=['object']).shape[1]}")
print(f"  - Data/Hora: {df_raw.select_dtypes(include=['datetime64']).shape[1]}")

### Análise de Valores Faltantes

**O que são valores faltantes?**
Valores faltantes (missing values) ocorrem quando uma informação não foi registrada para um paciente. Isso pode acontecer por:
- Teste não realizado
- Dados não coletados
- Erros de entrada

**Por que isso importa?**
Muitos algoritmos de aprendizado de máquina não conseguem trabalhar com valores faltantes diretamente. Precisamos decidir se vamos:
- Remover as linhas com valores faltantes
- Preencher com valores estimados (imputação)
- Remover colunas com muitos valores faltantes

In [ ]:
# Análise detalhada de valores faltantes
missing_data = pd.DataFrame({
    'Coluna': df_raw.columns,
    'Faltantes': df_raw.isnull().sum().values,
    'Percentual': (df_raw.isnull().sum() / len(df_raw) * 100).round(2).values,
    'Tipo': df_raw.dtypes.values
})

missing_data = missing_data.sort_values('Faltantes', ascending=False)

print("\nValores Faltantes por Variável (Top 20):")
print(missing_data.head(20).to_string(index=False))

# Estatísticas gerais
total_cells = df_raw.shape[0] * df_raw.shape[1]
missing_cells = df_raw.isnull().sum().sum()
missing_pct = missing_cells / total_cells * 100

print(f"\n\nEstatísticas Gerais:")
print(f"  - Total de células: {total_cells:,}")
print(f"  - Células faltantes: {missing_cells:,}")
print(f"  - Percentual de dados faltantes: {missing_pct:.2f}%")

In [ ]:
# Visualizar distribuição de valores faltantes
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Gráfico 1: Top 20 variáveis com mais valores faltantes
top_missing = missing_data.head(20)
axes[0].barh(range(len(top_missing)), top_missing['Percentual'])
axes[0].set_yticks(range(len(top_missing)))
axes[0].set_yticklabels(top_missing['Coluna'])
axes[0].set_xlabel('Percentual de Valores Faltantes (%)')
axes[0].set_title('Top 20 Variáveis com Mais Valores Faltantes', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

# Adicionar valores nas barras
for i, v in enumerate(top_missing['Percentual']):
    axes[0].text(v + 1, i, f'{v:.1f}%', va='center')

# Gráfico 2: Distribuição de completude por variável
completude = 100 - missing_data['Percentual']
axes[1].hist(completude, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Completude (%)')
axes[1].set_ylabel('Número de Variáveis')
axes[1].set_title('Distribuição da Completude das Variáveis', fontsize=12, fontweight='bold')
axes[1].axvline(completude.mean(), color='red', linestyle='--', linewidth=2, label=f'Média: {completude.mean():.1f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/01_missing_values_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico salvo em: ../figures/01_missing_values_analysis.png")

## Seção 4: Análise da Variável Alvo (Óbito)

**O que é a variável alvo?**
A variável alvo é aquilo que queremos prever. Neste caso, é se o paciente foi a óbito (1) ou sobreviveu (0).

**Por que analisar a variável alvo?**
Precisamos entender:
- Quantos pacientes foram a óbito vs. sobreviveram
- Se há desbalanceamento (muito mais de um tipo que do outro)
- Como isso afeta o treinamento do modelo

In [ ]:
# Análise da variável alvo na base bruta
print("\n" + "="*80)
print("ANÁLISE DA VARIÁVEL ALVO - BASE BRUTA")
print("="*80)

# Verificar se a coluna 'obito' existe
if 'obito' in df_raw.columns:
    obito_counts = df_raw['obito'].value_counts(dropna=False)
    obito_pct = df_raw['obito'].value_counts(normalize=True, dropna=False) * 100
    
    print("\nDistribuição de Óbitos:")
    resultado = pd.DataFrame({
        'Categoria': ['Sobreviveu (0)', 'Óbito (1)', 'Faltante'],
        'Contagem': [obito_counts.get(0, 0), obito_counts.get(1, 0), obito_counts.get(np.nan, 0)],
        'Percentual': [obito_pct.get(0, 0), obito_pct.get(1, 0), obito_pct.get(np.nan, 0)]
    })
    print(resultado.to_string(index=False))
    
    # Calcular taxa de mortalidade (excluindo faltantes)
    obito_valid = df_raw['obito'].dropna()
    taxa_mortalidade = (obito_valid == 1).sum() / len(obito_valid) * 100
    print(f"\nTaxa de Mortalidade (dados válidos): {taxa_mortalidade:.2f}%")
else:
    print("\n⚠ Coluna 'obito' não encontrada na base bruta.")
    print("Colunas disponíveis:", df_raw.columns.tolist())

In [ ]:
# Visualizar distribuição da variável alvo
if 'obito' in df_raw.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Gráfico 1: Contagem
    obito_counts = df_raw['obito'].value_counts(dropna=False)
    labels = ['Sobreviveu', 'Óbito', 'Faltante']
    colors = ['#2ecc71', '#e74c3c', '#95a5a6']
    
    axes[0].bar(range(len(obito_counts)), obito_counts.values, color=colors[:len(obito_counts)])
    axes[0].set_xticks(range(len(obito_counts)))
    axes[0].set_xticklabels([labels[i] for i in obito_counts.index if i == i or pd.isna(i)])
    axes[0].set_ylabel('Número de Pacientes')
    axes[0].set_title('Distribuição de Óbitos (Contagem)', fontsize=12, fontweight='bold')
    
    # Adicionar valores nas barras
    for i, v in enumerate(obito_counts.values):
        axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')
    
    # Gráfico 2: Percentual (excluindo faltantes)
    obito_valid = df_raw['obito'].dropna().value_counts()
    axes[1].pie(obito_valid.values, labels=['Sobreviveu', 'Óbito'], autopct='%1.1f%%',
                colors=['#2ecc71', '#e74c3c'], startangle=90)
    axes[1].set_title('Proporção de Óbitos (dados válidos)', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../figures/02_target_variable_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Gráfico salvo em: ../figures/02_target_variable_distribution.png")

## Seção 5: Análise de Variáveis Demográficas

**Variáveis demográficas** são características dos pacientes como idade, sexo e raça. Essas variáveis são importantes porque:
1. Podem influenciar o risco de óbito
2. Precisamos verificar se há vieses nos dados (ex: mais homens que mulheres)
3. Serão usadas para análise de justiça algorítmica

In [ ]:
print("\n" + "="*80)
print("ANÁLISE DE VARIÁVEIS DEMOGRÁFICAS - BASE BRUTA")
print("="*80)

# Idade
if 'idade' in df_raw.columns:
    print("\n1. IDADE")
    print(f"   - Mínima: {df_raw['idade'].min():.0f} anos")
    print(f"   - Máxima: {df_raw['idade'].max():.0f} anos")
    print(f"   - Média: {df_raw['idade'].mean():.1f} anos")
    print(f"   - Mediana: {df_raw['idade'].median():.1f} anos")
    print(f"   - Desvio padrão: {df_raw['idade'].std():.1f} anos")
    print(f"   - Faltantes: {df_raw['idade'].isnull().sum()}")

# Gênero
if 'genero' in df_raw.columns:
    print("\n2. GÊNERO")
    genero_counts = df_raw['genero'].value_counts(dropna=False)
    print(genero_counts)
    print(f"   - Faltantes: {df_raw['genero'].isnull().sum()}")

# Raça
if 'raca' in df_raw.columns:
    print("\n3. RAÇA/COR")
    raca_counts = df_raw['raca'].value_counts(dropna=False)
    print(raca_counts)
    print(f"   - Faltantes: {df_raw['raca'].isnull().sum()}")

In [ ]:
# Visualizar distribuição demográfica
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Distribuição de idade
if 'idade' in df_raw.columns:
    idade_valid = df_raw['idade'].dropna()
    axes[0, 0].hist(idade_valid, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[0, 0].axvline(idade_valid.mean(), color='red', linestyle='--', linewidth=2, label=f'Média: {idade_valid.mean():.1f}')
    axes[0, 0].axvline(idade_valid.median(), color='green', linestyle='--', linewidth=2, label=f'Mediana: {idade_valid.median():.1f}')
    axes[0, 0].set_xlabel('Idade (anos)')
    axes[0, 0].set_ylabel('Frequência')
    axes[0, 0].set_title('Distribuição de Idade', fontsize=12, fontweight='bold')
    axes[0, 0].legend()

# Gráfico 2: Box plot de idade
if 'idade' in df_raw.columns:
    axes[0, 1].boxplot(idade_valid, vert=True)
    axes[0, 1].set_ylabel('Idade (anos)')
    axes[0, 1].set_title('Box Plot de Idade', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)

# Gráfico 3: Distribuição de gênero
if 'genero' in df_raw.columns:
    genero_counts = df_raw['genero'].value_counts()
    axes[1, 0].bar(range(len(genero_counts)), genero_counts.values, color=['#3498db', '#e74c3c'])
    axes[1, 0].set_xticks(range(len(genero_counts)))
    axes[1, 0].set_xticklabels(genero_counts.index)
    axes[1, 0].set_ylabel('Número de Pacientes')
    axes[1, 0].set_title('Distribuição de Gênero', fontsize=12, fontweight='bold')
    
    # Adicionar valores
    for i, v in enumerate(genero_counts.values):
        axes[1, 0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Gráfico 4: Distribuição de raça
if 'raca' in df_raw.columns:
    raca_counts = df_raw['raca'].value_counts()
    axes[1, 1].barh(range(len(raca_counts)), raca_counts.values, color='coral')
    axes[1, 1].set_yticks(range(len(raca_counts)))
    axes[1, 1].set_yticklabels(raca_counts.index)
    axes[1, 1].set_xlabel('Número de Pacientes')
    axes[1, 1].set_title('Distribuição de Raça/Cor', fontsize=12, fontweight='bold')
    
    # Adicionar valores
    for i, v in enumerate(raca_counts.values):
        axes[1, 1].text(v + 50, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/03_demographic_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico salvo em: ../figures/03_demographic_analysis.png")

## Seção 6: Visão Geral da Base Tratada

### O que é a base tratada?
A base tratada é uma versão processada da base bruta, onde:
- Variáveis foram selecionadas (apenas 26 variáveis mais importantes)
- Dados foram limpos e validados
- Valores faltantes foram tratados
- Variáveis foram padronizadas (ex: nomes em inglês)

In [ ]:
print("\n" + "="*80)
print("BASE DE DADOS TRATADA - INFORMAÇÕES GERAIS")
print("="*80)

print(f"\nDimensões: {df_treated.shape[0]:,} pacientes × {df_treated.shape[1]} variáveis")
print(f"\nNomes das colunas:")
print(df_treated.columns.tolist())
print(f"\nPrimeiras 5 linhas:")
print(df_treated.head())

In [ ]:
# Análise de valores faltantes na base tratada
missing_treated = pd.DataFrame({
    'Coluna': df_treated.columns,
    'Faltantes': df_treated.isnull().sum().values,
    'Percentual': (df_treated.isnull().sum() / len(df_treated) * 100).round(2).values,
    'Tipo': df_treated.dtypes.values
})

missing_treated = missing_treated.sort_values('Faltantes', ascending=False)

print("\nValores Faltantes por Variável (Base Tratada):")
print(missing_treated[missing_treated['Faltantes'] > 0].to_string(index=False))

# Estatísticas gerais
total_cells = df_treated.shape[0] * df_treated.shape[1]
missing_cells = df_treated.isnull().sum().sum()
missing_pct = missing_cells / total_cells * 100

print(f"\n\nEstatísticas Gerais (Base Tratada):")
print(f"  - Total de células: {total_cells:,}")
print(f"  - Células faltantes: {missing_cells:,}")
print(f"  - Percentual de dados faltantes: {missing_pct:.2f}%")

In [ ]:
# Comparação entre base bruta e tratada
print("\n" + "="*80)
print("COMPARAÇÃO: BASE BRUTA vs. BASE TRATADA")
print("="*80)

comparacao = pd.DataFrame({
    'Métrica': [
        'Número de Pacientes',
        'Número de Variáveis',
        'Células Faltantes (%)',
        'Completude Média (%)'
    ],
    'Base Bruta': [
        f"{df_raw.shape[0]:,}",
        df_raw.shape[1],
        f"{(df_raw.isnull().sum().sum() / (df_raw.shape[0] * df_raw.shape[1]) * 100):.2f}%",
        f"{(100 - (df_raw.isnull().sum().sum() / (df_raw.shape[0] * df_raw.shape[1]) * 100)):.2f}%"
    ],
    'Base Tratada': [
        f"{df_treated.shape[0]:,}",
        df_treated.shape[1],
        f"{(df_treated.isnull().sum().sum() / (df_treated.shape[0] * df_treated.shape[1]) * 100):.2f}%",
        f"{(100 - (df_treated.isnull().sum().sum() / (df_treated.shape[0] * df_treated.shape[1]) * 100)):.2f}%"
    ]
})

print("\n" + comparacao.to_string(index=False))

## Seção 7: Análise Descritiva da Base Tratada

Vamos explorar as variáveis numéricas da base tratada com estatísticas descritivas.

In [ ]:
# Estatísticas descritivas
print("\nEstatísticas Descritivas - Base Tratada:")
print(df_treated.describe().round(2))

In [ ]:
# Visualizar distribuição de variáveis numéricas principais
numeric_cols = df_treated.select_dtypes(include=[np.number]).columns.tolist()

# Remover colunas que são variáveis alvo ou identificadores
plot_cols = [col for col in numeric_cols if col not in ['death', 'icu', 'mv']]

# Selecionar as primeiras 12 variáveis para visualização
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for idx, col in enumerate(plot_cols[:12]):
    data = df_treated[col].dropna()
    axes[idx].hist(data, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequência')
    axes[idx].set_title(f'Distribuição de {col}', fontsize=10, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/04_treated_data_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico salvo em: ../figures/04_treated_data_distributions.png")

## Seção 8: Análise da Variável Alvo na Base Tratada

In [ ]:
# Análise da variável alvo na base tratada
print("\n" + "="*80)
print("ANÁLISE DA VARIÁVEL ALVO - BASE TRATADA")
print("="*80)

if 'death' in df_treated.columns:
    death_counts = df_treated['death'].value_counts().sort_index()
    death_pct = df_treated['death'].value_counts(normalize=True).sort_index() * 100
    
    print("\nDistribuição de Óbitos:")
    resultado = pd.DataFrame({
        'Categoria': ['Sobreviveu (0)', 'Óbito (1)'],
        'Contagem': death_counts.values,
        'Percentual': death_pct.values.round(2)
    })
    print(resultado.to_string(index=False))
    
    taxa_mortalidade = death_pct[1]
    print(f"\nTaxa de Mortalidade: {taxa_mortalidade:.2f}%")
    print(f"\n⚠ NOTA IMPORTANTE: O desbalanceamento é importante!")
    print(f"   Se {taxa_mortalidade:.1f}% dos pacientes foram a óbito, o modelo pode simplesmente")
    print(f"   prever 'sobreviveu' para todos e ainda ter alta acurácia. Precisaremos")
    print(f"   usar métricas mais apropriadas como F1-score, AUC-ROC, etc.")

In [ ]:
# Visualizar distribuição da variável alvo na base tratada
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

death_counts = df_treated['death'].value_counts().sort_index()
labels = ['Sobreviveu', 'Óbito']
colors = ['#2ecc71', '#e74c3c']

# Gráfico 1: Contagem
axes[0].bar(range(len(death_counts)), death_counts.values, color=colors)
axes[0].set_xticks(range(len(death_counts)))
axes[0].set_xticklabels(labels)
axes[0].set_ylabel('Número de Pacientes')
axes[0].set_title('Distribuição de Óbitos - Base Tratada (Contagem)', fontsize=12, fontweight='bold')

# Adicionar valores nas barras
for i, v in enumerate(death_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Gráfico 2: Percentual
axes[1].pie(death_counts.values, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Proporção de Óbitos - Base Tratada', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/05_target_distribution_treated.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico salvo em: ../figures/05_target_distribution_treated.png")

## Seção 9: Correlações entre Variáveis

**O que é correlação?**
Correlação mede como duas variáveis se relacionam. Uma correlação:
- **Próxima de +1**: Quando uma aumenta, a outra também aumenta
- **Próxima de -1**: Quando uma aumenta, a outra diminui
- **Próxima de 0**: Não há relação linear

**Por que isso importa?**
- Variáveis muito correlacionadas podem causar multicolinearidade
- Ajuda a identificar variáveis redundantes
- Mostra quais variáveis estão relacionadas com a variável alvo

In [ ]:
# Calcular matriz de correlação
numeric_cols = df_treated.select_dtypes(include=[np.number]).columns
correlation_matrix = df_treated[numeric_cols].corr()

# Correlação com a variável alvo
if 'death' in correlation_matrix.columns:
    target_corr = correlation_matrix['death'].sort_values(ascending=False)
    
    print("\nCorrelação com a Variável Alvo (Óbito):")
    print(target_corr.round(3))

In [ ]:
# Visualizar matriz de correlação
fig, ax = plt.subplots(figsize=(14, 12))

# Criar heatmap
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)

ax.set_title('Matriz de Correlação - Base Tratada', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../figures/06_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico salvo em: ../figures/06_correlation_matrix.png")

## Seção 10: Análise de Outliers

**O que são outliers?**
Outliers são valores extremos que se desviam muito do padrão. Podem ser:
- Erros de entrada
- Casos genuinamente extremos
- Problemas de medição

**Por que detectar outliers?**
- Podem distorcer o treinamento do modelo
- Podem indicar dados problemáticos
- Precisamos decidir se removemos ou mantemos

In [ ]:
# Detectar outliers usando IQR (Interquartile Range)
def detect_outliers_iqr(data):
    """
    Detecta outliers usando o método IQR.
    
    Outliers são valores fora do intervalo [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
    """
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    return (data < lower_bound) | (data > upper_bound)

# Contar outliers por variável
numeric_cols = df_treated.select_dtypes(include=[np.number]).columns
outlier_counts = {}

for col in numeric_cols:
    outliers = detect_outliers_iqr(df_treated[col].dropna())
    outlier_counts[col] = outliers.sum()

outlier_df = pd.DataFrame({
    'Variável': list(outlier_counts.keys()),
    'Número de Outliers': list(outlier_counts.values()),
    'Percentual': [f"{(count / len(df_treated) * 100):.2f}%" for count in outlier_counts.values()]
}).sort_values('Número de Outliers', ascending=False)

print("\nDetecção de Outliers (Método IQR):")
print(outlier_df[outlier_df['Número de Outliers'] > 0].to_string(index=False))

In [ ]:
# Visualizar outliers com box plots
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

plot_cols = [col for col in numeric_cols if col not in ['death', 'icu', 'mv']]

for idx, col in enumerate(plot_cols[:12]):
    data = df_treated[col].dropna()
    axes[idx].boxplot(data, vert=True)
    axes[idx].set_ylabel(col)
    axes[idx].set_title(f'Box Plot de {col}', fontsize=10, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/07_outliers_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico salvo em: ../figures/07_outliers_boxplots.png")

## Seção 11: Resumo e Conclusões

### Principais Achados da Análise Exploratória

In [ ]:
print("\n" + "="*80)
print("RESUMO DA ANÁLISE EXPLORATÓRIA")
print("="*80)

print("""
1. DADOS BRUTOS:
   - 16.836 pacientes com 78 variáveis
   - 38.58% de dados faltantes (muito desafio!)
   - Muitas variáveis com >50% de faltantes
   - Requer limpeza e seleção significativa

2. DADOS TRATADOS:
   - 6.011 pacientes com 26 variáveis (seleção inteligente)
   - 13.12% de dados faltantes (muito melhor!)
   - Variáveis mais relevantes já selecionadas
   - Pronto para modelagem com tratamento menor

3. VARIÁVEL ALVO (ÓBITO):
   - Desbalanceamento presente (nem todos morrem)
   - Precisaremos de métricas apropriadas para desbalanceamento
   - F1-score, AUC-ROC são mais apropriados que acurácia

4. VARIÁVEIS DEMOGRÁFICAS:
   - Idade varia de 0 a 100+ anos
   - Distribuição de gênero pode ter vieses
   - Raça/cor tem muitos valores faltantes na base bruta
   - Importante para análise de justiça algorítmica

5. PRÓXIMOS PASSOS:
   - Pré-processamento: tratamento de faltantes, normalização
   - Feature engineering: criar novas variáveis se necessário
   - Modelagem: treinar LightGBM, XGBoost, CatBoost
   - Explicabilidade: análise SHAP
   - Justiça: verificar vieses algorítmicos
""")

print("\n" + "="*80)
print("Análise exploratória concluída! Prossiga para o notebook 02_preprocessing.ipynb")
print("="*80)

## Referências e Recursos Adicionais

- **Pandas Documentation**: https://pandas.pydata.org/docs/
- **Matplotlib**: https://matplotlib.org/
- **Seaborn**: https://seaborn.pydata.org/
- **Scikit-learn**: https://scikit-learn.org/

---

**Notebook criado para fins educacionais e de pesquisa.**
Data: Dezembro de 2025